# PDDL Attack Path Generation
Generates PDDL attack paths from CVE natural language descriptions using the `cve2pddlap` pipeline.

**Run order:** Cell 1 → 2 → 3(interactive preview) → 4 (LLM generation)

## 1. Environment Setup

In [1]:
import sys
import os

sys.path.insert(0, '../../src')

from cve2pddlap.core.data_loader import load_cve_list, load_few_shot_pool, select_few_shot_examples, FewShotExample
from cve2pddlap.core.prompt_builder import build_messages, ALL_SELECTABLE_NAMES, SMI_NAMES, OI_NAMES
from cve2pddlap.core.experiment_runner import run_single, run_batch
from cve2pddlap.llm_providers import create_provider, ALL_HOSTS, ALL_HOSTS_TIERED, MODEL_TIER
from cve2pddlap.utils.config import get_api_key, settings

print('Dependencies loaded successfully')
print(f'Selectable instructions: {ALL_SELECTABLE_NAMES}')
print(f'Available models: {ALL_HOSTS}')

Dependencies loaded successfully
Selectable instructions: ['SMI1', 'SMI2', 'SMI3', 'SMI4', 'SMI5', 'OI1', 'OI2', 'OI3']
Available models: ['claude', 'gpt4', 'deepseek_r1', 'qwen_max', 'mistral_large', 'groq_llama', 'qwen', 'deepseek', 'gemini', 'zhipu', 'mistral_small', 'groq_mixtral', 'deepseek_r1_ollama', 'qwen3b', 'qwen7b', 'qwen14b', 'qwen7b_coder']


## 2. Data

In [2]:
DATASET_PATH = '../../resources/data/CVE-PDDL-NNL-ReAP'
TARGET_POOL_PATH = '../../resources/data/target_pool.json'

# Reference dataset: all CVE/AP examples available for few-shot
few_shot_pool = load_few_shot_pool(DATASET_PATH)
pool_by_key = {ex.key: ex for ex in few_shot_pool}
print(f'Few-shot pool: {len(few_shot_pool)} examples ({len(set(ex.cve_id for ex in few_shot_pool))} CVEs)')

# Target pool: CVEs to generate attack paths for (edit target_pool.json to customise)
target_pool = load_cve_list(TARGET_POOL_PATH)
target_by_id = {entry.cve_id: entry for entry in target_pool}
target_cve_ids = [entry.cve_id for entry in target_pool]
print(f'Target pool: {len(target_pool)} CVEs')
for entry in target_pool:
    print(f'  {entry.cve_id}')

Few-shot pool: 55 examples (21 CVEs)
Target pool: 26 CVEs
  CVE-2025-66032
  CVE-2025-64755
  CVE-2025-59536
  CVE-2025-54795
  CVE-2025-9930
  CVE-2022-1471
  CVE-2022-40149
  CVE-2022-40150
  CVE-2023-2976
  CVE-2023-33202
  CVE-2023-34055
  CVE-2023-44487
  CVE-2023-46589
  CVE-2023-6378
  CVE-2024-12798
  CVE-2024-22243
  CVE-2024-22259
  CVE-2024-22262
  CVE-2024-34447
  CVE-2024-38286
  CVE-2024-38809
  CVE-2024-38816
  CVE-2024-38820
  CVE-2024-47072
  CVE-2025-22228
  CVE-2025-24813


## 3. Generate Domain PDDL
Select CVE, instructions, few-shot count, and specific few-shot examples to preview the rendered prompt.

In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Target CVE (from target_pool.json) ---
cve_dropdown = widgets.Dropdown(
    options=target_cve_ids,
    value=target_cve_ids[0] if target_cve_ids else None,
    description='Target CVE:',
    layout=widgets.Layout(width='380px')
)

# --- Model selector: tiered (Large/Mid/Small), separators have value=None ---
_DEFAULT_MODEL = settings.LLM.model_host
model_dropdown = widgets.Dropdown(
    options=ALL_HOSTS_TIERED,
    value=_DEFAULT_MODEL,
    description='Model:',
    layout=widgets.Layout(width='440px'),
    style={'description_width': '60px'},
)

def _on_model_change(change):
    """Prevent selecting separator headers — snap back to previous valid value."""
    if change['new'] is None:
        prev = change['old']
        if prev is not None:
            model_dropdown.value = prev
    _refresh_instructions()

model_dropdown.observe(_on_model_change, names='value')

# --- Decoding parameters ---
temp_slider = widgets.FloatSlider(
    value=0.0, min=0.0, max=2.0, step=0.1,
    description='Temperature:',
    layout=widgets.Layout(width='380px'),
    readout_format='.1f',
)
seed_input = widgets.IntText(
    value=42,
    description='Seed:',
    layout=widgets.Layout(width='200px'),
)
topp_slider = widgets.FloatSlider(
    value=1.0, min=0.0, max=1.0, step=0.05,
    description='Top-p:',
    layout=widgets.Layout(width='380px'),
    readout_format='.2f',
)
max_tokens_input = widgets.IntText(
    value=6144,
    description='Max tokens:',
    layout=widgets.Layout(width='200px'),
)

# --- Instruction selector (OI optional, SMI auto-managed by tier) ---
oi_select = widgets.SelectMultiple(
    options=ALL_SELECTABLE_NAMES,
    description='Extra instr:',
    rows=8,
    layout=widgets.Layout(width='240px'),
    style={'description_width': '80px'},
)

# Info box: shows what is always-on for current model + shot mode
instruction_info = widgets.HTML(value='')

def _refresh_instructions():
    """
    Auto-select SMI1-5 in oi_select when small model is chosen in few-shot mode.
    Large/mid models: SMI optional (not pre-selected).
    Zero-shot: SMI always on in template regardless, so no pre-selection needed.
    Updates the always-on info box.
    """
    host = model_dropdown.value
    n_shot = few_shot_slider.value
    tier = MODEL_TIER.get(host, 'large')
    is_fewshot = n_shot > 0

    # Determine which SMIs to auto-select
    if is_fewshot and tier == 'small':
        # Small model + few-shot: SMI1-5 must be selected (weaker models need guidance)
        auto_smi = tuple(SMI_NAMES)
    else:
        # Large/mid model, or zero-shot: SMI handled by template; leave as-is
        # Remove auto-selection but preserve any OI the user already picked
        auto_smi = ()

    # Merge auto_smi with whatever OI the user has manually selected
    current = set(oi_select.value)
    user_oi = current & set(OI_NAMES)
    oi_select.value = tuple(sorted(set(auto_smi) | user_oi,
                                   key=ALL_SELECTABLE_NAMES.index))

    # Build info HTML
    if is_fewshot:
        mandatory = 'BMI1-5'
        if tier == 'small':
            mandatory += ' + SMI1-5 (auto, small model)'
            note = 'Few-shot · small model: BMI6 + SMI1-5 forced into Extra instr'
        else:
            tier_label = 'Large' if tier == 'large' else 'Mid'
            note = f'Few-shot · {tier_label} model: BMI1-5 always on; SMI/OI optional'
    else:
        mandatory = 'BMI1-6 + SMI1-5'
        note = 'Zero-shot: BMI1-6 + SMI1-5 always on for ALL models; only OI optional'

    color = '#1a6b1a' if tier == 'large' else ('#7a4e00' if tier == 'mid' else '#8b0000')
    instruction_info.value = (
        f'<div style="padding:6px 10px;background:#f5f5f5;border-left:3px solid {color}'
        f';font-size:12px;color:#333">'
        f'<b>Always-on:</b> {mandatory}<br><span style="color:#666">{note}</span></div>'
    )

# --- Few-shot count slider ---
few_shot_slider = widgets.IntSlider(
    value=0, min=0, max=5, step=1,
    description='Few-shot:',
    layout=widgets.Layout(width='380px')
)

def _example_options(exclude_cve):
    return ['(random)'] + [k for k in sorted(pool_by_key) if not k.startswith(exclude_cve)]

# --- Dynamic few-shot example selectors (up to 5) ---
example_selectors = []
for i in range(5):
    sel = widgets.Dropdown(
        options=_example_options(cve_dropdown.value),
        value='(random)',
        description=f'Example {i+1}:',
        layout=widgets.Layout(width='380px'),
        disabled=True
    )
    example_selectors.append(sel)

example_box = widgets.VBox(example_selectors)

def on_slider_change(change):
    n = change['new']
    opts = _example_options(cve_dropdown.value)
    for i, sel in enumerate(example_selectors):
        sel.disabled = (i >= n)
        sel.options = opts
        if i >= n:
            sel.value = '(random)'
    _refresh_instructions()

def on_cve_change(change):
    opts = _example_options(change['new'])
    for sel in example_selectors:
        sel.options = opts
        sel.value = '(random)'

few_shot_slider.observe(on_slider_change, names='value')
cve_dropdown.observe(on_cve_change, names='value')

# Initial refresh
_refresh_instructions()

# --- Preview button ---
preview_button = widgets.Button(
    description='Preview Prompt',
    button_style='info',
    layout=widgets.Layout(width='160px')
)
output_area = widgets.Output()

def on_preview(b):
    with output_area:
        clear_output()
        cve_id = cve_dropdown.value
        selected_oi = list(oi_select.value)
        n_shot = few_shot_slider.value
        entry = target_by_id[cve_id]
        examples = None
        if n_shot > 0:
            keys = [sel.value for sel in example_selectors[:n_shot] if sel.value != '(random)']
            examples = [pool_by_key[k] for k in keys]
        msgs = build_messages(
            cve_id=cve_id,
            cve_description=entry.description,
            selected_oi=selected_oi,
            few_shot_examples=examples or [],
        )
        for msg in msgs:
            role = msg['role'].upper()
            print(f'--- {role} ---')
            print(msg['content'])
            print()

preview_button.on_click(on_preview)

display(widgets.VBox([
    widgets.HTML('<b>Target</b>'),
    cve_dropdown,
    widgets.HTML('<b>Model &nbsp;<small style="color:#888">(── separators not selectable)</small></b>'),
    model_dropdown,
    widgets.HTML('<b>Decoding</b>'),
    widgets.HBox([temp_slider, widgets.VBox([seed_input, max_tokens_input])]),
    topp_slider,
    widgets.HTML('<b>Instructions</b>'),
    instruction_info,
    widgets.HBox([oi_select, widgets.HTML(
        '<div style="font-size:11px;color:#555;padding:4px 8px;max-width:260px">'
        '<b>SMI1-5:</b> auto-selected for small models in few-shot mode<br>'
        '<b>OI1-3:</b> always optional, select manually<br><br>'
        'BMI1-6 are always on (not shown here)'
        '</div>'
    )]),
    widgets.HTML('<b>Few-shot Examples</b>'),
    few_shot_slider,
    example_box,
    widgets.HTML('<b>Preview</b>'),
    preview_button,
    output_area,
]))

In [4]:
# Check API key for the selected model
HOST = model_dropdown.value
if HOST is None:
    print('Please select a model first (not a section header).')
else:
    from cve2pddlap.llm_providers import MODEL_TIER
    tier = MODEL_TIER.get(HOST, 'unknown')
    if tier == 'small':
        print(f'Local model [{HOST}] — no API key needed.')
    else:
        from cve2pddlap.utils.config import get_api_key
        # Map host to provider name for API key lookup
        _host_to_provider = {
            'qwen_max': 'qwen',
            'deepseek_r1': 'deepseek',
            'mistral_small': 'mistral', 'mistral_large': 'mistral',
            'groq_llama': 'groq', 'groq_mixtral': 'groq',
        }
        provider = _host_to_provider.get(HOST, HOST)
        try:
            key = get_api_key(provider)
            print(f'[{HOST}] API key loaded: {key[:8]}...')
        except ValueError as e:
            print(f'Error: {e}')
            print(f'Please set the API key in .env')

[gpt4] API key loaded: sk-proj-...


In [5]:
# Generate the domain by LLM
# Read all configuration from the widgets above
TARGET_CVE = cve_dropdown.value
SELECTED_OI = list(oi_select.value)
N_SHOT = few_shot_slider.value
HOST = model_dropdown.value
TEMPERATURE = temp_slider.value
SEED = seed_input.value
TOP_P = topp_slider.value
MAX_TOKENS = max_tokens_input.value
SAVE_OUTPUT = True

if HOST is None:
    raise ValueError('Please select a model (not a section header) in the dropdown above.')

entry = target_by_id[TARGET_CVE]

# Resolve examples from selectors
examples = None
if N_SHOT > 0:
    chosen = []
    random_slots = []
    for i in range(N_SHOT):
        val = example_selectors[i].value
        if val != '(random)':
            chosen.append(pool_by_key[val])
        else:
            random_slots.append(i)
    if random_slots:
        exclude = {TARGET_CVE} | {ex.cve_id for ex in chosen}
        rand_pool = [ex for ex in few_shot_pool if ex.cve_id not in exclude]
        import random as _random
        rng = _random.Random(42)
        rand_picks = rng.sample(rand_pool, min(len(random_slots), len(rand_pool)))
        chosen.extend(rand_picks)
    examples = chosen[:N_SHOT]
    print(f'Few-shot examples: {[e.key for e in examples]}')

llm = create_provider(
    host=HOST,
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    seed=SEED,
    top_p=TOP_P,
)
print(f'Provider: {llm}')
print(f'Generating attack path for {TARGET_CVE}...')

result = run_single(
    cve_id=TARGET_CVE,
    cve_description=entry.description,
    llm=llm,
    selected_oi=SELECTED_OI,
    few_shot_examples=examples
)

print('\n===== Generated PDDL =====')
print(result)

if SAVE_OUTPUT:
    out_dir = '../../experiments'
    os.makedirs(out_dir, exist_ok=True)
    shot_label = f'{N_SHOT}shot'
    provider_name = HOST.replace('_', '-')
    out_file = os.path.join(out_dir, f'{TARGET_CVE}_{provider_name}_{shot_label}.pddl')
    with open(out_file, 'w') as f:
        f.write(result)
    print(f'Saved to: {out_file}')

Few-shot examples: ['CVE-2024-47072 / AP1']
Provider: GPT4Provider(model='gpt-4o-mini', temperature=0.0, seed=42, top_p=1.0)
Generating attack path for CVE-2025-66032...

===== Generated PDDL =====
(define (domain AED)
  (:requirements
    :adl
    :fluents
  )
  (:functions
    (total-cost)
    (version ?Software)
  )
  (:types
    attacker - actor
    user - actor
    target-system
    software
    cve-identifier
    exploit-technique - technique
    coding-tool - software
    context-window
    untrusted-content
  )
  (:constants
    CVE_2025_66032 - cve-identifier
    bypass-read-only-validation - exploit-technique
  )
  (:predicates
    (spoofing ?Target - target-system)
    (tampering ?Target - target-system)
    (repudiation ?Target - target-system)
    (information-disclosure ?Target - target-system)
    (denial-of-service ?Target - target-system)
    (elevation-of-privilege ?Target - target-system)
    (exposes-attack-surface ?Target - target-system ?CVEID - cve-identifier)
  

## 4. Generate Problem PDDL for Reference Domain
Generate problem.pddl from a **reference domain.pddl** (known-good) to validate the problem generation prompt.
Select any CVE/AP from the dataset, generate problem.pddl, then verify solvability in `evaluation.ipynb`.

In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from cve2pddlap.core.prompt_builder import build_problem_messages

# --- Reference AP selector (all examples with problem.pddl) ---
ref_ap_dropdown = widgets.Dropdown(
    options=[ex.key for ex in few_shot_pool if ex.problem_pddl],
    description='Reference AP:',
    layout=widgets.Layout(width='380px'),
    style={'description_width': '100px'},
)

# --- Model selector (independent from domain generation) ---
ref_model_dropdown = widgets.Dropdown(
    options=ALL_HOSTS_TIERED,
    value=settings.LLM.model_host,
    description='Model:',
    layout=widgets.Layout(width='440px'),
    style={'description_width': '60px'},
)

def _on_ref_model_change(change):
    if change['new'] is None:
        prev = change['old']
        if prev is not None:
            ref_model_dropdown.value = prev

ref_model_dropdown.observe(_on_ref_model_change, names='value')

# --- Decoding parameters ---
ref_temp_slider = widgets.FloatSlider(
    value=0.0, min=0.0, max=2.0, step=0.1,
    description='Temperature:',
    layout=widgets.Layout(width='380px'),
    readout_format='.1f',
)
ref_seed_input = widgets.IntText(
    value=42, description='Seed:',
    layout=widgets.Layout(width='200px'),
)
ref_topp_slider = widgets.FloatSlider(
    value=1.0, min=0.0, max=1.0, step=0.05,
    description='Top-p:',
    layout=widgets.Layout(width='380px'),
    readout_format='.2f',
)
ref_max_tokens = widgets.IntText(
    value=6144, description='Max tokens:',
    layout=widgets.Layout(width='200px'),
)

# --- Problem few-shot examples (exclude selected AP's CVE) ---
ref_problem_shot_slider = widgets.IntSlider(
    value=0, min=0, max=5, step=1,
    description='Few-shot:',
    layout=widgets.Layout(width='380px')
)

ref_problem_pool = [ex for ex in few_shot_pool if ex.problem_pddl]
ref_problem_pool_by_key = {ex.key: ex for ex in ref_problem_pool}

def _ref_problem_opts(exclude_cve):
    return ['(random)'] + [k for k in sorted(ref_problem_pool_by_key) if not k.startswith(exclude_cve)]

ref_problem_selectors = []
for i in range(5):
    sel = widgets.Dropdown(
        options=_ref_problem_opts(ref_ap_dropdown.value.split(' / ')[0]),
        value='(random)',
        description=f'Example {i+1}:',
        layout=widgets.Layout(width='380px'),
        disabled=True
    )
    ref_problem_selectors.append(sel)

def _on_ref_ap_change(change):
    cve = change['new'].split(' / ')[0]
    opts = _ref_problem_opts(cve)
    for sel in ref_problem_selectors:
        sel.options = opts
        sel.value = '(random)'

def _on_ref_shot_change(change):
    n = change['new']
    cve = ref_ap_dropdown.value.split(' / ')[0]
    opts = _ref_problem_opts(cve)
    for i, sel in enumerate(ref_problem_selectors):
        sel.disabled = (i >= n)
        sel.options = opts
        if i >= n:
            sel.value = '(random)'

ref_ap_dropdown.observe(_on_ref_ap_change, names='value')
ref_problem_shot_slider.observe(_on_ref_shot_change, names='value')

# --- Generate button ---
ref_gen_btn = widgets.Button(
    description='Generate Problem',
    button_style='success',
    layout=widgets.Layout(width='180px')
)
ref_gen_output = widgets.Output()

def _on_ref_gen(b):
    with ref_gen_output:
        clear_output()
        key = ref_ap_dropdown.value
        ref_ex = pool_by_key[key]
        cve_id = ref_ex.cve_id
        domain_pddl = ref_ex.domain_pddl

        # Resolve few-shot examples
        n = ref_problem_shot_slider.value
        p_examples = None
        if n > 0:
            chosen = []
            random_slots = []
            for i in range(n):
                val = ref_problem_selectors[i].value
                if val != '(random)':
                    chosen.append(ref_problem_pool_by_key[val])
                else:
                    random_slots.append(i)
            if random_slots:
                exclude = {cve_id} | {ex.cve_id for ex in chosen}
                pool = [ex for ex in ref_problem_pool if ex.cve_id not in exclude]
                import random as _random
                rng = _random.Random(42)
                picks = rng.sample(pool, min(len(random_slots), len(pool)))
                chosen.extend(picks)
            p_examples = chosen[:n]
            print(f'Few-shot examples: {[e.key for e in p_examples]}')

        HOST = ref_model_dropdown.value
        if HOST is None:
            print('ERROR: select a model (not a separator).')
            return

        ref_llm = create_provider(
            host=HOST,
            max_tokens=ref_max_tokens.value,
            temperature=ref_temp_slider.value,
            seed=ref_seed_input.value,
            top_p=ref_topp_slider.value,
        )

        msgs = build_problem_messages(
            cve_id=cve_id,
            domain_pddl=domain_pddl,
            few_shot_examples=p_examples,
        )

        print(f'Model: {ref_llm}')
        print(f'Generating problem.pddl for {key} (reference domain)...\n')
        gen_problem = ref_llm.send(msgs)

        print('===== Generated Problem PDDL =====')
        print(gen_problem)

        # Save to experiments/
        out_dir = '../../experiments'
        os.makedirs(out_dir, exist_ok=True)
        provider_name = HOST.replace('_', '-')
        n_label = f'{n}shot'
        safe_key = key.replace(' / ', '_').replace(' ', '')
        fname = f'{safe_key}_ref_{provider_name}_{n_label}_problem.pddl'
        out_path = os.path.join(out_dir, fname)
        with open(out_path, 'w') as f:
            f.write(gen_problem)
        print(f'\nSaved to: {out_path}')

ref_gen_btn.on_click(_on_ref_gen)

display(widgets.VBox([
    widgets.HTML('<b>Reference AP (known-good domain)</b>'),
    ref_ap_dropdown,
    widgets.HTML('<b>Model</b>'),
    ref_model_dropdown,
    widgets.HTML('<b>Decoding</b>'),
    widgets.HBox([ref_temp_slider, widgets.VBox([ref_seed_input, ref_max_tokens])]),
    ref_topp_slider,
    widgets.HTML('<b>Problem few-shot examples</b>'),
    ref_problem_shot_slider,
    widgets.VBox(ref_problem_selectors),
    ref_gen_btn,
    ref_gen_output,
]))

## 5. Generate Problem PDDL for Generated Domain
Generate problem.pddl from the **generated domain.pddl** produced in Cell 9.
Select few-shot examples (domain+problem pairs) independently from domain generation.

In [10]:
from cve2pddlap.core.prompt_builder import build_problem_messages
from cve2pddlap.core.data_loader import select_few_shot_examples

# Pool of examples that have problem.pddl
problem_pool = [ex for ex in few_shot_pool if ex.problem_pddl]
problem_pool_by_key = {ex.key: ex for ex in problem_pool}

def _problem_example_options(exclude_cve):
    return ['(random)'] + [k for k in sorted(problem_pool_by_key) if not k.startswith(exclude_cve)]

# --- Problem few-shot slider ---
problem_shot_slider = widgets.IntSlider(
    value=0, min=0, max=5, step=1,
    description='Few-shot:',
    layout=widgets.Layout(width='380px')
)

# --- Problem example selectors ---
problem_selectors = []
for i in range(5):
    sel = widgets.Dropdown(
        options=_problem_example_options(cve_dropdown.value),
        value='(random)',
        description=f'Example {i+1}:',
        layout=widgets.Layout(width='380px'),
        disabled=True
    )
    problem_selectors.append(sel)

def on_problem_slider_change(change):
    n = change['new']
    opts = _problem_example_options(cve_dropdown.value)
    for i, sel in enumerate(problem_selectors):
        sel.disabled = (i >= n)
        sel.options = opts
        if i >= n:
            sel.value = '(random)'

problem_shot_slider.observe(on_problem_slider_change, names='value')

# --- Preview button ---
problem_preview_btn = widgets.Button(
    description='Preview Problem Prompt',
    button_style='info',
    layout=widgets.Layout(width='200px')
)
problem_output = widgets.Output()

def _resolve_problem_examples():
    n = problem_shot_slider.value
    if n == 0:
        return None
    chosen = []
    random_slots = []
    for i in range(n):
        val = problem_selectors[i].value
        if val != '(random)':
            chosen.append(problem_pool_by_key[val])
        else:
            random_slots.append(i)
    if random_slots:
        exclude = {cve_dropdown.value} | {ex.cve_id for ex in chosen}
        rand_pool = [ex for ex in problem_pool if ex.cve_id not in exclude]
        import random as _random
        rng = _random.Random(42)
        rand_picks = rng.sample(rand_pool, min(len(random_slots), len(rand_pool)))
        chosen.extend(rand_picks)
    return chosen[:n]

def on_problem_preview(b):
    with problem_output:
        clear_output()
        p_examples = _resolve_problem_examples()
        if p_examples:
            print(f'Problem few-shot examples: {[e.key for e in p_examples]}\n')
        msgs = build_problem_messages(
            cve_id=TARGET_CVE,
            domain_pddl=result,
            few_shot_examples=p_examples,
        )
        total_chars = sum(len(m['content']) for m in msgs)
        print(f'Messages: {len(msgs)} | Est. prompt tokens: ~{total_chars // 4}\n')
        for msg in msgs:
            print(f"{'='*20} [{msg['role'].upper()}] {'='*20}")
            print(msg['content'])
            print()

problem_preview_btn.on_click(on_problem_preview)

display(
    widgets.VBox([
        widgets.HTML('<b>Problem PDDL Few-shot Examples (domain+problem pairs)</b>'),
        problem_shot_slider,
        widgets.Label('Drag slider to enable example selectors:'),
        widgets.VBox(problem_selectors),
        problem_preview_btn,
    ]),
    problem_output
)


Output()

In [11]:
# Generate problem.pddl based on the generated domain
# Uses TARGET_CVE, result, llm, N_SHOT, SAVE_OUTPUT from Cell 9

p_examples = _resolve_problem_examples()
if p_examples:
    print(f'Problem few-shot examples: {[e.key for e in p_examples]}')

problem_msgs = build_problem_messages(
    cve_id=TARGET_CVE,
    domain_pddl=result,
    few_shot_examples=p_examples,
)

print(f'Generating problem.pddl for {TARGET_CVE}...')
problem_result = llm.send(problem_msgs)

print('\n===== Generated Problem PDDL =====')
print(problem_result)

if SAVE_OUTPUT:
    provider_name = HOST.replace('_', '-')
    shot_label = f'{N_SHOT}shot'
    problem_file = os.path.join(out_dir, f'{TARGET_CVE}_{provider_name}_{shot_label}_problem.pddl')
    with open(problem_file, 'w') as f:
        f.write(problem_result)
    print(f'Saved to: {problem_file}')

Problem few-shot examples: ['CVE-2022-1471 / AP1']
Generating problem.pddl for CVE-2025-66032...

===== Generated Problem PDDL =====
(define (problem AEDI-elevation-of-privilege)
  (:domain AED)
  (:objects
    SEFA - target-system
    software_SEFA - software
    untrusted-content1 - untrusted-content
    context-window1 - context-window
    attacker1 - attacker
    user1 - user
  )
  (:init
    (= (total-cost) 0)
    (has-vulnerability SEFA CVE_2025_66032)
    (= (version software_SEFA) 1000001)
    (exposes-attack-surface SEFA CVE_2025_66032)
    (context-window-available SEFA context-window1)
    (has-sufficient-code-execution-privilege SEFA)
  )
  (:goal (and (elevation-of-privilege SEFA)))
  (:metric minimize (total-cost)))
Saved to: ../../experiments/CVE-2025-66032_gpt4_1shot_problem.pddl
